In [ ]:
# Cell 1: Auth + setup
import json, os, io, sys
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient().get_secret('GCP_SERVICE_ACCOUNT')
sa_path = '/kaggle/working/gcp_sa.json'
with open(sa_path, 'w') as f:
    f.write(secret)
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = sa_path

import subprocess
subprocess.run(['gcloud', 'auth', 'activate-service-account', '--key-file=/kaggle/working/gcp_sa.json', '--quiet'])

import pip
subprocess.run(['pip', 'install', '-q', 'google-cloud-storage==2.10.0', 'google-auth==2.22.0'])

from google.cloud import storage
client = storage.Client()
bucket = client.bucket('signbridge-data')
print('GCS auth OK')

In [ ]:
# Cell 2: Clone repo
import sys, subprocess
subprocess.run(['git', 'clone', 'https://github.com/Nosher007/signbridge.git', '/kaggle/working/signbridge'])
sys.path.insert(0, '/kaggle/working/signbridge')
print('Repo ready')

In [ ]:
# Cell 3: GPU check
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} -- GPUs: {gpus}')
assert len(gpus) > 0, 'No GPU! Enable T4 in Settings'

In [ ]:
# Cell 4: Load WLASL annotation + build video->gloss mapping
import json, numpy as np

nslt_bytes = bucket.blob('raw/wlasl/wlasl_data/nslt_100.json').download_as_bytes()
nslt = json.loads(nslt_bytes)

wlasl_bytes = bucket.blob('raw/wlasl/wlasl_data/WLASL_v0.3.json').download_as_bytes()
wlasl = json.loads(wlasl_bytes)

vid_to_gloss = {}
for entry in wlasl:
    gloss = entry['gloss']
    for inst in entry['instances']:
        vid_to_gloss[inst['video_id']] = gloss

records = []
for vid_id, info in nslt.items():
    gloss = vid_to_gloss.get(vid_id)
    if gloss is None:
        continue
    split = info.get('subset', 'train')
    records.append({'video_id': vid_id, 'gloss': gloss, 'split': split})

classes = sorted(set(r['gloss'] for r in records))
class_to_idx = {c: i for i, c in enumerate(classes)}
print(f'Total records: {len(records)}')
print(f'Classes: {len(classes)}')
print(f'Splits: {set(r["split"] for r in records)}')

In [ ]:
# Cell 5: MobileNetV2 feature extraction (~30-45 min)
import cv2, numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3)
)
base.trainable = False
print('Feature extractor ready')

SEQ_LEN = 30
IMG_SIZE = 224

def extract_features(video_bytes):
    with open('/tmp/_clip.mp4', 'wb') as f:
        f.write(video_bytes)
    cap = cv2.VideoCapture('/tmp/_clip.mp4')
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frames.append(frame)
    cap.release()
    if len(frames) == 0:
        return np.zeros((SEQ_LEN, 1280), dtype='float32')
    frames = np.array(frames, dtype='float32')
    T = len(frames)
    if T >= SEQ_LEN:
        idx = np.linspace(0, T-1, SEQ_LEN, dtype=int)
        frames = frames[idx]
    else:
        pad = np.zeros((SEQ_LEN - T, IMG_SIZE, IMG_SIZE, 3), dtype='float32')
        frames = np.concatenate([frames, pad], axis=0)
    frames = preprocess_input(frames)
    return base.predict(frames, verbose=0).astype('float32')

splits_data = {'train': [], 'val': [], 'test': []}
failed = 0
for i, rec in enumerate(records):
    vid_id = rec['video_id']
    split  = rec['split']
    label  = class_to_idx[rec['gloss']]
    blob_path = f"raw/wlasl/wlasl_data/videos/{vid_id}.mp4"
    try:
        video_bytes = bucket.blob(blob_path).download_as_bytes()
        feats = extract_features(video_bytes)
        splits_data[split].append((feats, label))
    except Exception as e:
        failed += 1
    if (i+1) % 100 == 0:
        print(f'  {i+1}/{len(records)} done, {failed} failed')

print(f'Extraction complete. Failed: {failed}')
for s, items in splits_data.items():
    print(f'  {s}: {len(items)} clips')

In [ ]:
# Cell 6: Save features to GCS
def save_npy_gcs(arr, blob_path):
    buf = io.BytesIO()
    np.save(buf, arr)
    buf.seek(0)
    bucket.blob(blob_path).upload_from_file(buf)
    print(f'  Saved {blob_path} -- {arr.shape}')

for split, items in splits_data.items():
    if len(items) == 0:
        continue
    X = np.array([x for x, _ in items], dtype='float32')
    y = np.array([l for _, l in items], dtype='int32')
    save_npy_gcs(X, f'processed/wlasl_mv2_features/X_{split}.npy')
    save_npy_gcs(y, f'processed/wlasl_mv2_features/y_{split}.npy')

save_npy_gcs(np.array(classes), 'processed/wlasl_mv2_features/classes.npy')
print('All features saved to GCS')

In [ ]:
# Cell 7: Train MobileNetV2+LSTM
from sklearn.utils.class_weight import compute_class_weight

def load_npy(blob_path):
    buf = io.BytesIO(bucket.blob(blob_path).download_as_bytes())
    return np.load(buf)

X_train = load_npy('processed/wlasl_mv2_features/X_train.npy')
y_train = load_npy('processed/wlasl_mv2_features/y_train.npy')
X_val   = load_npy('processed/wlasl_mv2_features/X_val.npy')
y_val   = load_npy('processed/wlasl_mv2_features/y_val.npy')
X_test  = load_npy('processed/wlasl_mv2_features/X_test.npy')
y_test  = load_npy('processed/wlasl_mv2_features/y_test.npy')
num_classes = len(classes)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

y_train_oh = tf.keras.utils.to_categorical(y_train, num_classes)
y_val_oh   = tf.keras.utils.to_categorical(y_val,   num_classes)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(30, 1280)),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax'),
], name='mobilenetv2_lstm')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

ckpt  = tf.keras.callbacks.ModelCheckpoint('/tmp/mv2_lstm_best.keras', save_best_only=True, monitor='val_loss', verbose=1)
es    = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
rlrop = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)

history = model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=50, batch_size=32,
    class_weight=class_weights,
    callbacks=[ckpt, es, rlrop]
)
print(f"Best val accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

In [ ]:
# Cell 8: Evaluate + save
from sklearn.metrics import f1_score
import time, subprocess

model = tf.keras.models.load_model('/tmp/mv2_lstm_best.keras')
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

top1 = np.mean(y_pred == y_test)
top5 = np.mean([y_test[i] in np.argsort(y_pred_prob[i])[-5:] for i in range(len(y_test))])
f1   = f1_score(y_test, y_pred, average='macro', zero_division=0)

t0 = time.time()
for _ in range(100):
    model.predict(X_test[:1], verbose=0)
latency = (time.time() - t0) / 100 * 1000

print(f'Top-1 Accuracy : {top1*100:.2f}%')
print(f'Top-5 Accuracy : {top5*100:.2f}%')
print(f'Macro F1       : {f1:.4f}')
print(f'Latency        : {latency:.1f} ms/sequence')

subprocess.run(['gsutil', 'cp', '/tmp/mv2_lstm_best.keras',
                'gs://signbridge-data/models/wlasl_mobilenetv2_lstm_v1.keras'], check=True)
print('Model saved to GCS')